# D20 S3 A2 - NumPy Image Augmentation and Metrics
Implements zero-filled translations, horizontal flipping, dataset augmentation, accuracy, confusion matrix, recall, and macro-F1.


In [1]:
import numpy as np
from sklearn.datasets import load_digits

digits = load_digits()
images = digits.images
labels = digits.target


In [2]:
def shift_image(img, dy, dx):
    img = np.asarray(img)
    height, width = img.shape
    shifted = np.zeros_like(img)

    if abs(dy) >= height or abs(dx) >= width:
        return shifted

    src_top = max(0, -dy)
    src_bottom = min(height, height - dy)
    src_left = max(0, -dx)
    src_right = min(width, width - dx)

    dst_top = max(0, dy)
    dst_bottom = min(height, height + dy)
    dst_left = max(0, dx)
    dst_right = min(width, width + dx)

    shifted[dst_top:dst_bottom, dst_left:dst_right] = img[src_top:src_bottom, src_left:src_right]
    return shifted


def flip_image(img):
    return np.asarray(img)[:, ::-1].copy()


def augment_dataset(X, y):
    X = np.asarray(X)
    y = np.asarray(y)
    down = np.array([shift_image(img, 1, 0) for img in X])
    up = np.array([shift_image(img, -1, 0) for img in X])
    right = np.array([shift_image(img, 0, 1) for img in X])
    left = np.array([shift_image(img, 0, -1) for img in X])
    X_aug = np.concatenate([X.copy(), down, up, right, left], axis=0)
    y_aug = np.concatenate([y.copy(), y.copy(), y.copy(), y.copy(), y.copy()])
    return X_aug, y_aug


def accuracy(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    if y_true.size == 0:
        return 0.0
    return float(np.mean(y_true == y_pred))


def confusion_matrix(y_true, y_pred, n_classes):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    matrix = np.zeros((n_classes, n_classes), dtype=int)
    for true_class, predicted_class in zip(y_true, y_pred):
        matrix[true_class, predicted_class] += 1
    return matrix


def per_class_recall(y_true, y_pred, n_classes):
    matrix = confusion_matrix(y_true, y_pred, n_classes)
    recalls = np.zeros(n_classes, dtype=float)
    for c in range(n_classes):
        actual_count = matrix[c].sum()
        if actual_count != 0:
            recalls[c] = matrix[c, c] / actual_count
    return recalls


def macro_f1(y_true, y_pred, n_classes):
    matrix = confusion_matrix(y_true, y_pred, n_classes)
    scores = np.zeros(n_classes, dtype=float)
    for c in range(n_classes):
        tp = matrix[c, c]
        fp = matrix[:, c].sum() - tp
        fn = matrix[c, :].sum() - tp
        denominator = 2 * tp + fp + fn
        if denominator != 0:
            scores[c] = 2 * tp / denominator
    return float(scores.mean())


In [3]:
print(shift_image([[1,2],[3,4]], 0, 1))
print(flip_image([[1,2,3],[4,5,6]]))
Xa, ya = augment_dataset(images[:2], labels[:2])
print(Xa.shape, ya.shape)
print(np.array_equal(ya[:2], labels[:2]))

y_true = [0,0,1,1,2]
y_pred = [0,0,1,2,2]
print(accuracy(y_true, y_pred))
print(per_class_recall(y_true, y_pred, 3))
print(round(macro_f1(y_true, y_pred, 3), 4))


[[0 1]
 [0 3]]
[[3 2 1]
 [6 5 4]]
(10, 8, 8) (10,)
True
0.8
[1.  0.5 1. ]
0.7778


In [4]:
# Real augmentation robustness comparison
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

X_train, X_test, y_train, y_test = train_test_split(
    images, labels, test_size=0.25, random_state=42, stratify=labels
)
X_train_aug, y_train_aug = augment_dataset(X_train, y_train)
X_test_shifted = np.array([shift_image(img, 1, 0) for img in X_test])

baseline = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, random_state=42))
augmented = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, random_state=42))
baseline.fit(X_train.reshape(len(X_train), -1), y_train)
augmented.fit(X_train_aug.reshape(len(X_train_aug), -1), y_train_aug)

baseline_shifted_accuracy = baseline.score(X_test_shifted.reshape(len(X_test_shifted), -1), y_test)
augmented_shifted_accuracy = augmented.score(X_test_shifted.reshape(len(X_test_shifted), -1), y_test)
augmented_clean_accuracy = augmented.score(X_test.reshape(len(X_test), -1), y_test)
print('Baseline shifted accuracy:', baseline_shifted_accuracy)
print('Augmented shifted accuracy:', augmented_shifted_accuracy)
print('Augmented clean accuracy:', augmented_clean_accuracy)
print('Shifted improvement:', augmented_shifted_accuracy - baseline_shifted_accuracy)


Baseline shifted accuracy: 0.36444444444444446
Augmented shifted accuracy: 0.7622222222222222
Augmented clean accuracy: 0.8666666666666667
Shifted improvement: 0.3977777777777778


In [5]:
# Checks corresponding to all 8 autograder requirements
sample = np.array([[1,2],[3,4]])
assert np.array_equal(shift_image(sample,0,1), np.array([[0,1],[0,3]]))
assert np.array_equal(shift_image(sample,1,0), np.array([[0,0],[1,2]]))
for dy, dx in [(2,0),(-1,3),(20,-20)]:
    assert shift_image(sample,dy,dx).shape == sample.shape

flipped = flip_image(np.array([[1,2,3],[4,5,6]]))
assert np.array_equal(flipped, np.array([[3,2,1],[6,5,4]]))
assert np.array_equal(flip_image(flipped), np.array([[1,2,3],[4,5,6]]))

small_X, small_y = images[:4], labels[:4]
Xa, ya = augment_dataset(small_X, small_y)
assert Xa.shape == (20,8,8) and ya.shape == (20,)
assert np.array_equal(Xa[:4], small_X)
for block in range(5):
    assert np.array_equal(ya[block*4:(block+1)*4], small_y)

assert augmented_shifted_accuracy - baseline_shifted_accuracy >= 0.05
assert augmented_clean_accuracy >= 0.80

expected_cm = np.array([[2,0,0],[0,1,1],[0,0,1]])
assert np.array_equal(confusion_matrix(y_true,y_pred,3), expected_cm)
assert np.isclose(accuracy(y_true,y_pred),0.8)
assert np.allclose(per_class_recall(y_true,y_pred,3),[1.0,0.5,1.0])
assert np.isclose(macro_f1(y_true,y_pred,3),7/9)
assert accuracy([],[]) == 0.0

print('All 8 autograder-style checks passed.')


All 8 autograder-style checks passed.
